# GridSpark AI – Notebook 02: Sentinel-2 Pre-fire Features

**Window:** 2020-08-20 to 2020-09-06 (pre-fire)  
**AOI:** 5 km buffer around Holiday Farm Fire ignition anchor  
**Indices:**
- **NDVI** – Normalized Difference Vegetation Index (B8/B4, 10 m native)
- **NDMI** – Normalized Difference Moisture Index (B8/B11, 10–20 m)
- **NBR** – Normalized Burn Ratio (B8/B12, 10–20 m)

> **Note on resolution:** Sentinel-2 B8 (NIR) is natively 10 m; B11 and B12 (SWIR) are 20 m. When computing NDMI and NBR, bands are harmonized at 20 m unless resampled. Report as '10–20 m Sentinel-2 indices.'

In [ ]:
import sys
sys.path.insert(0, '../scripts')

from config import (
    IGNITION_LAT, IGNITION_LON,
    S2_START_DATE, S2_END_DATE, S2_CLOUD_PCT_MAX,
    S2_FEATURES_OUT,
)

print(f'Ignition anchor: ({IGNITION_LAT}, {IGNITION_LON})')
print(f'Pre-fire window: {S2_START_DATE} to {S2_END_DATE}')
print(f'Max cloud cover: {S2_CLOUD_PCT_MAX}%')

## Option A – Google Earth Engine Python API

Requires authentication: `earthengine authenticate`

If you have not authenticated, skip to **Option B** (GEE Code Editor script) below.

In [ ]:
try:
    import ee
    ee.Initialize(project='your-gee-project-id')  # Replace with your GEE project
    GEE_AVAILABLE = True
    print('Earth Engine initialized.')
except Exception as e:
    GEE_AVAILABLE = False
    print(f'GEE not available: {e}')
    print('Falling back to GEE Code Editor script (Option B).')

In [ ]:
if GEE_AVAILABLE:
    # Define geometry
    ignition = ee.Geometry.Point([IGNITION_LON, IGNITION_LAT])
    aoi = ignition.buffer(5000)

    # Filter Sentinel-2 collection
    s2 = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filterDate(S2_START_DATE, S2_END_DATE)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', S2_CLOUD_PCT_MAX))
        .sort('CLOUDY_PIXEL_PERCENTAGE')
    )

    count = s2.size().getInfo()
    print(f'Sentinel-2 scenes matching criteria: {count}')

    if count == 0:
        print('WARNING: No scenes found. Try relaxing cloud cover threshold.')
    else:
        img = ee.Image(s2.first())
        date = img.date().format('YYYY-MM-dd').getInfo()
        cloud = img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
        print(f'Best scene: {date}  Cloud cover: {cloud:.1f}%')

In [ ]:
if GEE_AVAILABLE:
    img = ee.Image(s2.first())

    # Compute indices
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndmi = img.normalizedDifference(['B8', 'B11']).rename('NDMI')  # 10–20 m
    nbr  = img.normalizedDifference(['B8', 'B12']).rename('NBR')   # 10–20 m

    stack = ndvi.addBands(ndmi).addBands(nbr)

    # Compute mean statistics over the 5 km AOI
    stats = stack.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=aoi,
        scale=20,
        maxPixels=1e8,
    ).getInfo()

    print('Pre-fire index statistics (5 km AOI):')
    for k, v in stats.items():
        print(f'  {k}: {v:.4f}' if v is not None else f'  {k}: None')

    # Save summary
    import pandas as pd
    from pathlib import Path
    Path(S2_FEATURES_OUT).parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame([{
        'date': img.date().format('YYYY-MM-dd').getInfo(),
        'cloud_pct': img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo(),
        'aoi_buffer_m': 5000,
        **{k: round(v, 4) if v is not None else None for k, v in stats.items()}
    }])
    df.to_csv(S2_FEATURES_OUT, index=False)
    print(f'\nSaved to {S2_FEATURES_OUT}')

## Option B – GEE Code Editor Script

If GEE Python API is unavailable, copy the code below into the [GEE Code Editor](https://code.earthengine.google.com/).

In [ ]:
gee_js = """
// GridSpark AI – Holiday Farm Fire Pre-fire Sentinel-2 Features
// Paste this into: https://code.earthengine.google.com/

var ignition = ee.Geometry.Point([-122.231, 44.172]);
var aoi = ignition.buffer(5000);

var s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
  .filterBounds(aoi)
  .filterDate("2020-08-20", "2020-09-07")
  .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
  .sort("CLOUDY_PIXEL_PERCENTAGE");

print("Scenes found:", s2.size());
print("Best scene date:", ee.Image(s2.first()).date().format("YYYY-MM-dd"));
print("Cloud cover:", ee.Image(s2.first()).get("CLOUDY_PIXEL_PERCENTAGE"));

var img = ee.Image(s2.first());

// NDVI: B8 (NIR, 10 m) / B4 (Red, 10 m)
var ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI");

// NDMI: B8 (NIR, 10 m) / B11 (SWIR-1, 20 m) → reported at 10–20 m
var ndmi = img.normalizedDifference(["B8", "B11"]).rename("NDMI");

// NBR: B8 (NIR, 10 m) / B12 (SWIR-2, 20 m) → reported at 10–20 m
var nbr = img.normalizedDifference(["B8", "B12"]).rename("NBR");

Map.centerObject(aoi, 11);
Map.addLayer(ndvi.clip(aoi), {min: 0, max: 1, palette: ["red","yellow","green"]}, "NDVI");
Map.addLayer(ndmi.clip(aoi), {min: -0.5, max: 0.5, palette: ["red","white","blue"]}, "NDMI");
Map.addLayer(nbr.clip(aoi), {min: -0.5, max: 1,   palette: ["red","white","green"]}, "NBR");
Map.addLayer(ignition, {color: "red"}, "Ignition anchor");

// AOI boundary
Map.addLayer(ee.Image().paint(aoi, 0, 3), {palette: ["red"]}, "5 km AOI");

// Print mean index values within AOI
var stack = ndvi.addBands(ndmi).addBands(nbr);
var stats = stack.reduceRegion({
  reducer: ee.Reducer.mean(),
  geometry: aoi,
  scale: 20,
  maxPixels: 1e8
});
print("Mean index values (5 km AOI, 20 m scale):", stats);

// Export mean stats as CSV to Google Drive
Export.table.toDrive({
  collection: ee.FeatureCollection([ee.Feature(aoi, stats)]),
  description: "gridspark_ai_sentinel2_prefire_features",
  fileFormat: "CSV",
  folder: "GridSparkAI"
});
"""

print('GEE Code Editor Script:')
print('=' * 60)
print(gee_js)

## Option C – Load pre-saved features (if already exported)

After running the GEE script, download the CSV from Google Drive and place it at:
`data/processed/sentinel2_prefire_features.csv`

In [ ]:
import pandas as pd
from pathlib import Path

if Path(S2_FEATURES_OUT).exists():
    df = pd.read_csv(S2_FEATURES_OUT)
    print('Loaded pre-saved Sentinel-2 features:')
    display(df)
else:
    print(f'No pre-saved features found at {S2_FEATURES_OUT}')
    print('Run GEE script (Option B) and export CSV to this path.')

    # Create placeholder with expected schema
    placeholder = pd.DataFrame([{
        'date': '2020-08-XX',
        'cloud_pct': None,
        'aoi_buffer_m': 5000,
        'NDVI_mean': None,
        'NDMI_mean': None,
        'NBR_mean': None,
        'NDVI_stdDev': None,
        'NDMI_stdDev': None,
        'NBR_stdDev': None,
        'note': 'Placeholder — run GEE script to populate'
    }])
    Path(S2_FEATURES_OUT).parent.mkdir(parents=True, exist_ok=True)
    placeholder.to_csv(S2_FEATURES_OUT, index=False)
    print(f'Placeholder schema saved to {S2_FEATURES_OUT}')
    display(placeholder)

---

## Scientific notes

1. **NDVI** (Normalized Difference Vegetation Index): Measures vegetation greenness and density. Pre-fire NDVI is a proxy for live fuel load. Higher NDVI = more vegetation = higher potential fuel.

2. **NDMI** (Normalized Difference Moisture Index): Measures vegetation water content using SWIR. Lower NDMI indicates drier fuels. Strongly correlated with fire weather and curing.

3. **NBR** (Normalized Burn Ratio): Pre-fire NBR combined with post-fire NBR gives the dNBR burn severity metric (MTBS standard). Pre-fire NBR alone is also a fuel proxy.

4. **Band resolution caveat:** B11 and B12 are 20 m; B8 is 10 m. Mixed-resolution indices are labeled '10–20 m' to avoid overstating spatial precision.

5. **Temporal window:** The 2020-08-20 to 2020-09-06 window was chosen to capture the driest pre-fire fuel state. Labor Day 2020 had multiple Oregon fires due to extreme wind events.